In [8]:
import numpy as np
from scipy.optimize import differential_evolution, minimize_scalar


In [9]:
def _solve_two_type_problem(
    V, lambda_1, lambda_2, theta_1, theta_2, *,
    decentralized, eps=1e-9, ftol=1e-11, maxiter=1000,
    initial_point=None,
):
    """Optimize the paper's exponential-service model, not an empirical M/G/1.

    Eliminate sojourn times exactly at fixed depths, then search the two
    remaining variables. Converged searches are numerical evidence, not a
    formal global-optimality certificate.
    """
    rates = np.asarray([lambda_1, lambda_2], dtype=float)
    costs = np.asarray([theta_1, theta_2], dtype=float)
    if not np.all(np.isfinite(np.r_[rates, costs])) or np.any(np.r_[rates, costs] <= 0):
        raise ValueError("Arrival rates and delay costs must be finite and positive.")
    if decentralized and not costs[0] > costs[1]:
        raise ValueError("The paper's IC formulation requires theta_1 > theta_2.")
    if not (0 < eps < 1) or not np.isfinite(ftol) or ftol <= 0:
        raise ValueError("Require 0 < eps < 1 and a positive finite ftol.")
    if not isinstance(maxiter, (int, np.integer)) or maxiter < 1:
        raise ValueError("maxiter must be a positive integer.")
    l1, l2 = rates

    def value(x):
        result = np.asarray(V(float(x)), dtype=float)
        if result.size != 1 or not np.all(np.isfinite(result)):
            raise ValueError("V must return one finite value for each nonnegative depth.")
        return float(result.item())

    if abs(value(0.0)) > 1e-12:
        raise ValueError("The paper requires V(0) = 0.")

    def allocation(x, mode):
        """Exact minimum-delay-cost allocation for these service depths."""
        x1, x2 = x
        p, q = rates * x
        rho = p + q
        if rho >= 1 or np.any(x < 0):
            return None
        if x1 == 0 and x2 == 0:
            return np.zeros(2)
        if x1 == 0:
            return np.array([0.0, x2 / (1 - q)])
        if x2 == 0:
            # An excluded type has x=w=0. Serving only the impatient type
            # cannot satisfy w1 <= w2 when the patient type takes that option.
            return None if mode == "IC" else np.array([x1 / (1 - p), 0.0])

        a, b = x1 / (1 - p), x2 / (1 - q)
        K = (p * x1 + q * x2) / (1 - rho)
        first1 = np.array([a, x2 / (1 - rho) + l1 * x1**2 / ((1 - p) * (1 - rho))])
        first2 = np.array([x1 / (1 - rho) + l2 * x2**2 / ((1 - q) * (1 - rho)), b])
        if mode == "1first":
            return first1
        if mode == "2first":
            return first2

        # Vertices of the achievable region intersected with w1 <= w2.
        # Do not restrict the decentralized problem to a pure priority order:
        # equality or extra delay may be needed for incentive compatibility.
        candidates = [np.array([a, max(a, first1[1])])]
        if first2[0] <= b:
            candidates.append(first2)
        equal = max(a, b, K / rho)
        candidates.append(np.array([equal, equal]))
        return min(candidates, key=lambda w: float(np.dot(rates * costs, w)))

    def evaluate(x, mode):
        w = allocation(x, mode)
        if w is None:
            return None
        benefit = np.array([value(x[0]), value(x[1])])
        welfare = float(np.dot(rates, benefit - costs * w))
        return welfare, w

    def depths(z):
        # Bounded coordinates: total load and the type-1 fraction of load.
        rho, share = z
        return rho * np.array([share, 1 - share]) / rates

    x0 = None
    if initial_point is not None:
        y0 = np.asarray(initial_point, dtype=float)
        if y0.shape != (4,) or not np.all(np.isfinite(y0)) or np.any(y0 < 0):
            raise ValueError("initial_point must contain four finite nonnegative entries.")
        loads = rates * y0[:2]
        rho0 = float(loads.sum())
        if rho0 > 1 - eps:
            raise ValueError("initial_point must satisfy the stability margin.")
        x0 = [rho0, loads[0] / rho0 if rho0 > 0 else 0.5]

    # Closure is always feasible and prevents a spurious negative optimum.
    candidates = [(0.0, np.zeros(2), np.zeros(2), "closed", None)]
    modes = ("IC",) if decentralized else ("1first", "2first")
    for mode in modes:
        def objective(z):
            result = evaluate(depths(z), mode)
            return 1e100 if result is None else -result[0]

        # Search both centralized priority branches explicitly. Two fixed
        # seeds per branch protect against dependence on one initialization.
        for seed in (1729, 20260908):
            res = differential_evolution(
                objective, bounds=[(0.0, 1 - eps), (0.0, 1.0)],
                seed=seed, x0=x0, popsize=20, maxiter=maxiter,
                tol=ftol, atol=ftol, polish=True,
            )
            if not res.success:
                raise RuntimeError(
                    f"{mode} search did not converge (seed={seed}): {res.message}. "
                    "Increase maxiter; no welfare table should be reported from this run."
                )
            x = depths(res.x)
            result = evaluate(x, mode)
            if result is not None:
                welfare, w = result
                candidates.append((welfare, x, w, mode, res))

    # Solve admissible one-type boundaries explicitly; random search need
    # not land exactly on zero service depth.
    for i in ((1,) if decentralized else (0, 1)):
        def single_objective(rho):
            x = rho / rates[i]
            return -rates[i] * (value(x) - costs[i] * x / (1 - rho))

        res = minimize_scalar(
            single_objective, bounds=(0.0, 1 - eps), method="bounded",
            options={"xatol": ftol, "maxiter": maxiter},
        )
        if not res.success:
            raise RuntimeError(f"Single-type search did not converge: {res.message}")
        x = np.zeros(2)
        x[i] = res.x / rates[i]
        welfare, w = evaluate(x, modes[0])
        candidates.append((welfare, x, w, f"only_type_{i + 1}", res))

    welfare, x, w, mode, raw_result = max(candidates, key=lambda item: item[0])
    loads = rates * x
    rho = float(loads.sum())
    own_lower = x / (1 - loads)
    K = float(np.dot(loads, x) / (1 - rho))
    slacks = {
        "stability": float(1 - eps - rho),
        "own_service_1": float(w[0] - own_lower[0]),
        "own_service_2": float(w[1] - own_lower[1]),
        "conservation": float(np.dot(loads, w) - K),
    }
    if decentralized:
        slacks["IC"] = float(w[1] - w[0])
    if rho >= 1 or not np.all(np.isfinite(np.r_[x, w, welfare])):
        raise RuntimeError("Optimization returned an invalid allocation.")
    if any(slack < -1e-8 for slack in slacks.values()):
        raise RuntimeError(f"Optimization returned an infeasible allocation: {slacks}")
    if np.any(w[x == 0] != 0):
        raise RuntimeError("Excluded types must have zero sojourn time.")

    return {
        "x_1": float(x[0]), "x_2": float(x[1]),
        "w_1": float(w[0]), "w_2": float(w[1]),
        "rho_1": float(loads[0]), "rho_2": float(loads[1]),
        "load": rho, "welfare": welfare, "slacks": slacks,
        "success": True,
        "message": "Best feasible result of converged multi-seed searches and explicit boundaries.",
        "global_optimality_certified": False,
        "allocation_regime": mode, "raw_result": raw_result,
    }


def solve_centralized_problem(
    V, lambda_1, lambda_2, theta_1, theta_2, *,
    eps=1e-9, ftol=1e-11, maxiter=1000, initial_point=None,
):
    return _solve_two_type_problem(
        V, lambda_1, lambda_2, theta_1, theta_2,
        decentralized=False, eps=eps, ftol=ftol, maxiter=maxiter,
        initial_point=initial_point,
    )


In [10]:
def solve_decentralized_problem(
    V, lambda_1, lambda_2, theta_1, theta_2, *,
    eps=1e-9, ftol=1e-11, maxiter=1000,
):
    return _solve_two_type_problem(
        V, lambda_1, lambda_2, theta_1, theta_2,
        decentralized=True, eps=eps, ftol=ftol, maxiter=maxiter,
    )


In [11]:
def compute_welfare_gap(
    V, lambda_1, lambda_2, theta_1, theta_2, *,
    eps=1e-9, ftol=1e-11, maxiter=1000,
):
    centralized = solve_centralized_problem(
        V, lambda_1, lambda_2, theta_1, theta_2,
        eps=eps, ftol=ftol, maxiter=maxiter,
    )
    decentralized = solve_decentralized_problem(
        V, lambda_1, lambda_2, theta_1, theta_2,
        eps=eps, ftol=ftol, maxiter=maxiter,
    )
    W_C, W_D = centralized["welfare"], decentralized["welfare"]
    tolerance = 1e-8 * max(1.0, abs(W_C), abs(W_D))
    if W_D > W_C + tolerance:
        raise RuntimeError(
            f"Decentralized welfare ({W_D}) exceeds centralized welfare ({W_C}). "
            "The numerical searches disagree; do not report this result."
        )
    if W_C <= 0:
        welfare_ratio, welfare_gap = np.nan, 0.0  # Paper's zero-welfare convention.
    elif W_D > W_C:
        welfare_ratio, welfare_gap = 1.0, 0.0  # Roundoff only, checked above.
    else:
        welfare_ratio = W_D / W_C
        welfare_gap = 1.0 - welfare_ratio

    return {
        "centralized_welfare": W_C, "decentralized_welfare": W_D,
        "welfare_ratio": welfare_ratio, "welfare_gap": welfare_gap,
        "centralized_solution": centralized, "decentralized_solution": decentralized,
    }


In [12]:
def lambdas_from_ratio(ratio, lambda_total):
    if ratio <= 0:
        raise ValueError("lambda_2/lambda_1 must be positive.")
    lambda_1 = lambda_total / (1.0 + ratio)
    lambda_2 = lambda_total * ratio / (1.0 + ratio)
    return float(lambda_1), float(lambda_2)


def compute_welfare_gap_table(
    value_specs,
    ratio_grid,
    *,
    lambda_total,
    theta_1,
    theta_2,
    verbose=True,
):
    rows = []
    for value_spec in value_specs:
        for ratio in ratio_grid:
            lambda_1, lambda_2 = lambdas_from_ratio(ratio, lambda_total)
            out = compute_welfare_gap(
                value_spec["fn"],
                lambda_1,
                lambda_2,
                theta_1,
                theta_2,
            )
            row = {
                "value_key": value_spec.get("key", value_spec.get("label", "V")),
                "value_label": value_spec.get("label", value_spec.get("key", "V")),
                "ratio": float(ratio),
                "lambda_1": lambda_1,
                "lambda_2": lambda_2,
                "centralized_welfare": out["centralized_welfare"],
                "decentralized_welfare": out["decentralized_welfare"],
                "welfare_ratio": out["welfare_ratio"],
                "welfare_gap": out["welfare_gap"],
                "centralized_solution": out["centralized_solution"],
                "decentralized_solution": out["decentralized_solution"],
            }
            rows.append(row)
            if verbose:
                print(
                    f"{row['value_key']:>10s}, "
                    f"ratio={ratio:>7g}, "
                    f"gap={100 * row['welfare_gap']:.3f}%"
                )
    return rows


def welfare_gap_matrix(rows, value_specs, ratio_grid, metric="welfare_gap"):
    matrix = np.full((len(value_specs), len(ratio_grid)), np.nan)
    for i, value_spec in enumerate(value_specs):
        key = value_spec.get("key", value_spec.get("label", "V"))
        for j, ratio in enumerate(ratio_grid):
            matches = [
                row for row in rows
                if row["value_key"] == key and np.isclose(row["ratio"], ratio)
            ]
            if matches:
                matrix[i, j] = matches[0][metric]
    return matrix

In [13]:
theta_1_fixed = 0.51
theta_2_fixed = 0.49
lambda_total_fixed = 1.0

value_specs = [
    {"key": "linear", "label": r"$V(x)=2x$", "fn": lambda x: 2 * x},
    {"key": "sqrt", "label": r"$V(x)=2\sqrt{x}$", "fn": lambda x: 2 * np.sqrt(np.maximum(x, 0.0))},
    {"key": "log", "label": r"$V(x)=2\log(1+x)$", "fn": lambda x: 2 * np.log1p(np.maximum(x, 0.0))},
    {"key": "exp", "label": r"$V(x)=2(1-e^{-x})$", "fn": lambda x: 2 * (1.0 - np.exp(-np.maximum(x, 0.0)))},
]

ratio_grid = np.array([0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0])

welfare_gap_rows = compute_welfare_gap_table(
    value_specs,
    ratio_grid,
    lambda_total=lambda_total_fixed,
    theta_1=theta_1_fixed,
    theta_2=theta_2_fixed,
    verbose=True,
)

welfare_gap = welfare_gap_matrix(welfare_gap_rows, value_specs, ratio_grid, metric="welfare_gap")
welfare_ratio = welfare_gap_matrix(welfare_gap_rows, value_specs, ratio_grid, metric="welfare_ratio")

print("\nWelfare gap 1 - W_D/W_C (percent):")
for spec, row in zip(value_specs, welfare_gap):
    print(f"{spec['key']:>8s}: {np.round(100 * row, 3)}")

print("\nLaTeX table rows (one-decimal percentages):")
for spec, row in zip(value_specs, welfare_gap):
    print(spec["label"] + " & " + " & ".join(f"{100 * gap:.1f}\\%" for gap in row) + r"\\")


    linear, ratio=    0.5, gap=0.000%
    linear, ratio=      1, gap=0.000%
    linear, ratio=      2, gap=3.706%
    linear, ratio=      4, gap=8.887%
    linear, ratio=      8, gap=12.541%
    linear, ratio=     16, gap=14.834%
    linear, ratio=     32, gap=16.151%
      sqrt, ratio=    0.5, gap=0.000%
      sqrt, ratio=      1, gap=0.000%
      sqrt, ratio=      2, gap=0.000%
      sqrt, ratio=      4, gap=0.452%
      sqrt, ratio=      8, gap=0.595%
      sqrt, ratio=     16, gap=0.506%
      sqrt, ratio=     32, gap=0.343%
       log, ratio=    0.5, gap=0.000%
       log, ratio=      1, gap=0.000%
       log, ratio=      2, gap=0.182%
       log, ratio=      4, gap=1.531%
       log, ratio=      8, gap=1.900%
       log, ratio=     16, gap=1.586%
       log, ratio=     32, gap=1.065%
       exp, ratio=    0.5, gap=0.000%
       exp, ratio=      1, gap=0.000%
       exp, ratio=      2, gap=0.000%
       exp, ratio=      4, gap=0.777%
       exp, ratio=      8, gap=1.019%
       ex

In [14]:
lambda_1_fixed = 0.1
lambda_2_fixed = 0.9

theta_pairs = [
    (0.51, 0.49),
    (0.52, 0.48),
    (0.55, 0.45),
    (0.6, 0.4),
]

theta_pair_rows = []
for spec in value_specs:
    row = {"value_key": spec["key"]}
    for theta_1_value, theta_2_value in theta_pairs:
        out = compute_welfare_gap(
            spec["fn"],
            lambda_1_fixed,
            lambda_2_fixed,
            theta_1_value,
            theta_2_value,
        )
        column_name = rf"$\theta_1={theta_1_value},\theta_2={theta_2_value}$"
        row[column_name] = out["welfare_gap"]
    theta_pair_rows.append(row)

theta_pair_gap_table = theta_pair_rows

print(f"Fixed lambda_1={lambda_1_fixed}, lambda_2={lambda_2_fixed}")
print("Welfare gap 1 - W_D/W_C for different theta pairs:")
for row in theta_pair_gap_table:
    formatted = {
        key: (f"{100 * value:.3f}%" if isinstance(value, float) else value)
        for key, value in row.items()
    }
    print(formatted)

print("\nLaTeX table rows (one-decimal percentages):")
for spec, row in zip(value_specs, theta_pair_rows):
    gaps = [value for key, value in row.items() if key != "value_key"]
    print(spec["label"] + " & " + " & ".join(f"{100 * gap:.1f}\\%" for gap in gaps) + r"\\")


Fixed lambda_1=0.1, lambda_2=0.9
Welfare gap 1 - W_D/W_C for different theta pairs:
{'value_key': 'linear', '$\\theta_1=0.51,\\theta_2=0.49$': '13.017%', '$\\theta_1=0.52,\\theta_2=0.48$': '11.108%', '$\\theta_1=0.55,\\theta_2=0.45$': '5.528%', '$\\theta_1=0.6,\\theta_2=0.4$': '0.000%'}
{'value_key': 'sqrt', '$\\theta_1=0.51,\\theta_2=0.49$': '0.592%', '$\\theta_1=0.52,\\theta_2=0.48$': '0.272%', '$\\theta_1=0.55,\\theta_2=0.45$': '0.000%', '$\\theta_1=0.6,\\theta_2=0.4$': '0.000%'}
{'value_key': 'log', '$\\theta_1=0.51,\\theta_2=0.49$': '1.881%', '$\\theta_1=0.52,\\theta_2=0.48$': '1.011%', '$\\theta_1=0.55,\\theta_2=0.45$': '0.000%', '$\\theta_1=0.6,\\theta_2=0.4$': '0.000%'}
{'value_key': 'exp', '$\\theta_1=0.51,\\theta_2=0.49$': '1.006%', '$\\theta_1=0.52,\\theta_2=0.48$': '0.296%', '$\\theta_1=0.55,\\theta_2=0.45$': '0.000%', '$\\theta_1=0.6,\\theta_2=0.4$': '0.000%'}

LaTeX table rows (one-decimal percentages):
$V(x)=2x$ & 13.0\% & 11.1\% & 5.5\% & 0.0\%\\
$V(x)=2\sqrt{x}$ & 0.6\